In [ ]:
# %matplotlib ipympl

In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
import scienceplots
import matplotlib as mpl

plt.style.use(['science','ieee'])
mpl.rcParams['path.simplify'] = True
mpl.rcParams['path.simplify_threshold'] = 1.0

In [ ]:
plt.get_backend()

In [ ]:
from src.base.lit_module.components.metrics_standard import MyMultiClassAccuracy, MyLoss
from src.base.lit_module.components.metric_gch import GCH_Metric, RGCH_Metric
from src.class_imbalance_ratios.analysis.summarise_metrics import MetricMetadata

In [ ]:
from src.class_imbalance_ratios.analysis.post_proc_conversions import delete_all_non_extension, delete_all_pt_extension
from src.class_imbalance_ratios.analysis.post_proc_conversions import convert_all_pt_metrics_to_parquet, convert_pt_class_counts_to_parquet

In [ ]:
from src.class_imbalance_ratios.analysis.metric_run_dir_analysis import agg_save_metrics, subgroup_save_acc
from src.class_imbalance_ratios.analysis.needed_paths import cls_imb_outputs_dir
from src.class_imbalance_ratios.analysis.needed_names import corr_store_names

In [ ]:
import logging

log_file = Path(os.getcwd()) / "src/log_analysis.log"
logging.basicConfig(filename=log_file, level=logging.DEBUG)

In [ ]:

display("=========== DELETING ALL PARQUET FILES EXPECTED BUT MISSING .PARQUET ===========")
delete_all_non_extension(outputs_dir=cls_imb_outputs_dir)

In [ ]:
display("=========== CONVERTING ALL .PT files to .PARQUET ===========")
convert_all_pt_metrics_to_parquet(outputs_dir=cls_imb_outputs_dir) # Took 30m 50.7s on CIF10 cls_imb locally

In [ ]:
display("=========== STORING SUMMARY GRADIENT CORRELATION METRICS ===========")
coupled_metric_paths = [
    dict(
        paths= f"run*/train_{corr_store_names["normed"]}.parquet", 
        prefix="Normalised-GC", 
        summary_mode_extras={
            "none": True,
            "cols": True,
            "both": True,
            "epochs": True,
            "subgroup": {
                "epochs": True,
                "steps": True, # Either steps and/or epochs can be True. Setting both False will lead to Warning. 
                "given_groups": {
                    "majority": [],
                    "minority": [],
                }
            }
        }
    ),
    dict(
        paths = f"run*/train_{corr_store_names["raw"]}.parquet", 
        prefix="Unnormalised-GC", # For prefix which has GC majority, minority are sourced with path
        summary_mode_extras={
            "none": True,
            "cols": True,
            "both": True,
            "epochs": True,
            "subgroup": {
                "epochs": True,
                "given_groups": {
                    "majority": [],
                    "minority": [],
                }
            }
        }
    )
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=None,

    run_paths_info=coupled_metric_paths
)

In [ ]:
plt.close('all')

In [ ]:
"""
TODO: 
- 
- Figure out none/both options for accuracy unless you can label with a legend of the "super groups" they belong to, so this does need specialised splitting, read Pandas book
    - For now they only need different markers actually per group. Aggregating groups separately comes after. 
    - Personally I would encourage simply processing and creating and storing a derived metric where we capture majority vs minority accuracy over a run. 
    - Then this can be run with the advantages of any other aggregation. Nothing more complicated I will accept really. Visualization pipeline will then be able to plot whatever is needed. Whether validation or train data i.e. list of the extra type and core metric "name". They occupy same plot. 
    - The difficulty is if I think unify train vs val either by color. But then common e.g. "name" e.g. major class, needs shared color. 
    - Figure out if above will work same for other metrics. 
"""
display("=========== STORING SUMMARY ACCURACY METRICS ===========") 
coupled_metric_paths = [
    dict(paths= f"run*/train_acc.parquet", prefix="Train-Accuracy", summary_mode_extras=["none", "both", "epochs", "cols"]),
    dict(paths = f"run*/val_acc.parquet", prefix="Validation-Accuracy", summary_mode_extras=["none", "both", "epochs", "cols"]) # TODO: When visualizing you can superimpose similar aggregates to one plot  # TODO  We may consider decoupling atypical measures that aren't traiditional aggregation across runs e.g. subgrouping - recommended
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=None,
    run_paths_info=coupled_metric_paths
)
subgroup_save_acc(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=None,
    run_paths_info=coupled_metric_paths,
)

In [ ]:
plt.close('all')

In [ ]:
display("=========== STORING SUMMARY LOSS METRICS ===========")
coupled_metric_paths = [
    dict(paths= f"run*/train_loss.parquet", prefix="Train-Loss", summary_mode_extras=["none", "epochs"]),
    dict(paths = f"run*/val_loss.parquet", prefix="Validation-Loss", summary_mode_extras=["none", "epochs"]) # TODO: When visualizing you can superimpose similar aggregates to one plot
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=None,

    run_paths_info=coupled_metric_paths
)

In [ ]:
plt.close('all')